# Phase 13: Crop Classifier Model Export to ONNX

This notebook loads the trained EfficientNet-B0 PyTorch crop disease classifier (`best_model.pth`), restores the weights state, traces it using a mock input tensor, compiles it to **ONNX format (`classifier.onnx`)**, and writes the label map and metadata descriptors inside the `exports/onnx/` bundling folder.

In [1]:
# Setup paths and imports
from pathlib import Path
import sys
import os
import json
import torch
import torch.nn as nn
import onnx

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name != 'ai' and PROJECT_ROOT.parent != PROJECT_ROOT:
    if (PROJECT_ROOT / 'ai').exists():
        PROJECT_ROOT = PROJECT_ROOT / 'ai'
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import utils

In [ ]:
# Build network and load best weights
device = torch.device("cpu")

def build_classifier(num_classes):
    import torchvision.models as models
    model = models.efficientnet_b0()
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)
    return model

model = build_classifier(utils.config.NUM_CLASSES).to(device)
best_pth = utils.paths.CLASSIFIER_MODEL_DIR / "best_model.pth"

if best_pth.exists():
    checkpoint = torch.load(str(best_pth), map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    print("✅ Loaded model weights successfully.")
else:
    print("⚠️ Warning: Best model weights checkpoint not found. Using random weights.")

In [ ]:
# Export classifier.onnx with dynamic batch axes
dest_path = utils.paths.CLASSIFIER_MODEL_DIR / "classifier.onnx"
export_path = utils.paths.PROJECT_ROOT / "exports" / "onnx" / "classifier.onnx"
export_path.parent.mkdir(parents=True, exist_ok=True)

dummy_input = torch.randn(1, 3, utils.config.IMAGE_SIZE, utils.config.IMAGE_SIZE)

print("🚀 Exporting PyTorch model to ONNX format...")
torch.onnx.export(
    model,
    dummy_input,
    str(dest_path),
    export_params=True,
    opset_version=12,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}
)

# Ensure all weights are fully embedded inside a single binary (avoiding split .data files)
try:
    model_proto = onnx.load(str(dest_path))
    onnx.save(model_proto, str(dest_path))
    # Clean up separate sidecar file if it was created
    data_sidecar = dest_path.with_suffix(".onnx.data")
    if data_sidecar.exists():
        data_sidecar.unlink()
except Exception as e:
    print(f"[WARNING] Failed to embed weights: {e}")

# Copy to exports directory
import shutil
shutil.copy(str(dest_path), str(export_path))
print("✅ Model successfully exported to ONNX format:", export_path)

In [ ]:
# Generate labels.json and metadata.json descriptors
labels_file = utils.paths.PROJECT_ROOT / "exports" / "onnx" / "labels.json"
metadata_file = utils.paths.PROJECT_ROOT / "exports" / "onnx" / "metadata.json"

# Load class labels from backend services folder
class_names_path = utils.paths.PROJECT_ROOT.parent / "backend" / "services" / "class_names.json"
if class_names_path.exists():
    with open(class_names_path, 'r') as f:
        labels = json.load(f)
else:
    labels = [f"Class {i}" for i in range(utils.config.NUM_CLASSES)]

with open(labels_file, 'w') as f:
    json.dump(labels, f, indent=4)

metadata = {
    "model_name": "EfficientNet-B0 Crop Disease Classifier",
    "input_shape": [1, 3, utils.config.IMAGE_SIZE, utils.config.IMAGE_SIZE],
    "input_name": "input",
    "output_name": "output",
    "normalization": {
        "mean": [0.485, 0.456, 0.406],
        "std": [0.229, 0.224, 0.225]
    },
    "num_classes": utils.config.NUM_CLASSES
}

with open(metadata_file, 'w') as f:
    json.dump(metadata, f, indent=4)

print("✅ Labels JSON created at:", labels_file)
print("✅ Metadata JSON created at:", metadata_file)